# Module 03 — Descriptive Statistics (240-student classroom data)
Dedicated notebook for Module 03 — reproduces every figure quoted in this module's slides/readings: missing data, gender × teaching-method crosstab, central tendency & dispersion, boxplot by school.

Human-reviewed English translation of the original Vietnamese notebook (code and results are identical, only text is translated).

SPSS/PSPP equivalents: `FREQUENCIES`, `CROSSTABS`, `EXAMINE`.

In [ ]:
# Run this cell first (Colab: click ▶). No installation needed.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"] = (7, 4)
pd.set_option("display.precision", 3)

def make_data(seed=2026, n=240):
    """SIMULATED 'Classroom of 240 students' dataset — SAME seed=2026 used
    throughout Modules 03-08, exactly matching lop_hoc_240.csv/.sav in the
    repo (not real data, but 100% reproducible)."""
    rng = np.random.default_rng(seed)
    school = rng.choice(list("ABC"), n, p=[.35, .35, .30])
    gender = rng.choice(["Male", "Female"], n)
    method = rng.choice(["Traditional", "Project-based"], n)
    study_hours = np.clip(rng.gamma(4, 1.2, n), 0.5, 15).round(1)
    interest = rng.normal(0, 1, n) + 0.15 * (method == "Project-based")
    anxiety = rng.normal(0, 1, n) - 0.25 * interest
    def likert(lat, load, noise=0.7):
        return np.clip(np.round(3 + load * lat + rng.normal(0, noise, n)), 1, 5).astype(int)
    d = pd.DataFrame({"id": np.arange(1, n + 1), "school": school, "gender": gender, "method": method,
                      "study_hours": study_hours})
    for k, (lat, load) in enumerate([(interest, .8), (interest, .7), (interest, .75)], 1):
        d[f"h{k}"] = likert(lat, load)
    for k, (lat, load) in enumerate([(anxiety, .8), (anxiety, .75), (anxiety, .7)], 1):
        d[f"a{k}"] = likert(lat, load)
    eff = d.school.map({"A": 3, "B": 0, "C": -3}).to_numpy()
    d["pretest"] = (rng.normal(60, 10, n) + eff).round(1)
    d["posttest"] = np.clip(d.pretest + 3 + 5 * (method == "Project-based") + rng.normal(0, 6, n), 0, 100).round(1)
    d["math"] = np.clip(35 + 2.5 * study_hours + 4 * interest - 3 * anxiety + eff + rng.normal(0, 6, n), 0, 100).round(1)
    d.loc[6, "study_hours"] = 48.0
    d.loc[[11, 57, 130], "h2"] = np.nan
    d["h2"] = d["h2"].astype("Int64")
    return d

df = make_data()
print("Created df:", df.shape)

## 1. Missing data — checking `h2`
SPSS equivalent: `Analyze > Descriptive Statistics > Frequencies`, look at the "Missing" row.

🎯 **What question does this answer?** Before computing ANY statistic, you need to know what percentage of data is missing and whether there's a pattern to it (MCAR/MAR/MNAR) — because how you handle missing data (dropping every case vs. only when needed) directly affects the reliability of EVERY analysis that follows.

In [ ]:
missing_h2 = df["h2"].isna().sum()
pct = missing_h2 / len(df) * 100
print(f"Missing cells in h2: {missing_h2} / {len(df)} ({pct:.2f}%)")

#### 📤 Actual output
`3 / 240 (1.25%)`. ✅ Matches the figure quoted in the lesson: a very small missing rate with no obvious pattern → pairwise handling is the most sensible choice for correlations involving `h2`, rather than dropping all 3 students from every analysis.

## 2. Crosstab: gender × teaching method
SPSS equivalent: `Analyze > Descriptive Statistics > Crosstabs`, tick `Cells... > Row percentages`.

🎯 **What question does this method answer?** A crosstab shows the JOINT DISTRIBUTION of two nominal/ordinal variables — used to spot BY EYE whether there might be a relationship between them, before running a formal test (chi-square, Module 07). The crosstab alone does NOT tell you whether that relationship is statistically significant.

In [ ]:
ct_count = pd.crosstab(df["gender"], df["method"])
ct_pct = pd.crosstab(df["gender"], df["method"], normalize="index") * 100
display(ct_count)
display(ct_pct.round(1))

#### 📤 Actual output
Male (n=121): Project-based 63 (52.1%), Traditional 58 (47.9%). Female (n=119): Project-based 52 (43.7%), Traditional 67 (56.3%). ✅ Matches the "crosstab trap" example in the lesson: the raw counts 63>52 feel like a bigger gap than it is — the row percentages show a smaller actual difference (52.1% vs 43.7%).

## 3. Central tendency & dispersion — Math scores
SPSS equivalent: `Analyze > Descriptive Statistics > Frequencies`, tick Mean/Median/Mode/Std deviation/Variance/Range.

In [ ]:
desc = df["math"].agg(["mean","median","std","var","min","max"])
mode_val = df["math"].mode().iloc[0]
print(desc.round(3))
print("Mode:", mode_val)
print(f"Range = {df['math'].max() - df['math'].min():.1f}")

#### 📤 Actual output
Mean ≈ 47.91, SD ≈ 9.68 — these two numbers become the inputs for the confidence interval in Module 04.

## 4. Illustrating "same mean, very different SD" (Cohen et al., pp.762–764)
Three artificial sets, all with mean = 6, but very different SD — why a mean alone is never enough to report.

In [ ]:
set1 = np.array([1, 2, 3, 4, 20])          # strong outlier
set2 = np.array([4, 5, 6, 6, 9])           # moderate spread
set3 = np.array([5, 6, 6, 6, 7])           # very tightly clustered
for name, s in [("Set 1 (has an outlier)", set1), ("Set 2", set2), ("Set 3 (clustered)", set3)]:
    print(f"{name}: mean={s.mean():.2f}, SD={s.std(ddof=1):.2f}")

#### 📤 Actual output
All three sets have mean=6.00 but SD is roughly 7.91 / 1.87 / 0.71 respectively — set (1)'s SD is 11x set (3)'s, despite an identical mean. ✅ Matches the illustration in the lesson.

## 5. Boxplot of Math scores by school
SPSS equivalent: `Analyze > Descriptive Statistics > Explore`, Plots > Boxplot.

🎯 **What question does this method answer?** A boxplot answers: is the distribution symmetric, are there outliers, how do the groups compare — reading the shape is much faster than reading raw numbers in a table.

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=df, x="school", y="math", order=["A","B","C"], ax=ax)
ax.set_title("Math scores by school (n=240)")
plt.show()
df.groupby("school")["math"].agg(["count","mean","std"]).round(2)

#### 📤 Actual output
School A (n=80, mean≈50.67) is highest, School C (n=61, mean≈45.34) is lowest — the same data used again in Module 06 (ANOVA F(2,237)=5.87, p=0.003).